In [ ]:
import pandas as pd
import os
from ucimlrepo import fetch_ucirepo 

# 1. Fetch the dataset (ID 10 = Automobile, ID 149 = Statlog Vehicles)
vehicle_data = fetch_ucirepo(id=10) 

# Extract features and targets as pandas DataFrames
X = vehicle_data.data.features 
y = vehicle_data.data.targets 

# Combine into a single DataFrame for EDA
df = pd.concat([X, y], axis=1)

# 2. Structural Review
print("Dataset Dimensions:", df.shape)
print("\n--- Data Types & Non-Null Counts ---")
df.info()

print("\n--- First 5 Rows ---")
display(df.head())

# 3. Save to Raw Directory
# Ensure the directory exists, then save the immutable raw data
os.makedirs('../data/raw', exist_ok=True)
df.to_csv('../data/raw/raw_vehicles.csv', index=False)
print("\nRaw data saved to data/raw/raw_vehicles.csv")

Dataset Dimensions: (205, 26)

--- Data Types & Non-Null Counts ---
<class 'pandas.DataFrame'>
RangeIndex: 205 entries, 0 to 204
Data columns (total 26 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   price              201 non-null    float64
 1   highway-mpg        205 non-null    int64  
 2   city-mpg           205 non-null    int64  
 3   peak-rpm           203 non-null    float64
 4   horsepower         203 non-null    float64
 5   compression-ratio  205 non-null    float64
 6   stroke             201 non-null    float64
 7   bore               201 non-null    float64
 8   fuel-system        205 non-null    str    
 9   engine-size        205 non-null    int64  
 10  num-of-cylinders   205 non-null    int64  
 11  engine-type        205 non-null    str    
 12  curb-weight        205 non-null    int64  
 13  height             205 non-null    float64
 14  width              205 non-null    float64
 15  length           

,price,highway-mpg,city-mpg,peak-rpm,horsepower,compression-ratio,stroke,bore,fuel-system,engine-size,...,wheel-base,engine-location,drive-wheels,body-style,num-of-doors,aspiration,fuel-type,make,normalized-losses,symboling
0,13495.0,27,21,5000.0,111.0,9.0,2.68,3.47,mpfi,130,...,88.6,front,rwd,convertible,2.0,std,gas,alfa-romero,NaN,3
1,16500.0,27,21,5000.0,111.0,9.0,2.68,3.47,mpfi,130,...,88.6,front,rwd,convertible,2.0,std,gas,alfa-romero,NaN,3
2,16500.0,26,19,5000.0,154.0,9.0,3.47,2.68,mpfi,152,...,94.5,front,rwd,hatchback,2.0,std,gas,alfa-romero,NaN,1
3,13950.0,30,24,5500.0,102.0,10.0,3.40,3.19,mpfi,109,...,99.8,front,fwd,sedan,4.0,std,gas,audi,164.0,2
4,17450.0,22,18,5500.0,115.0,8.0,3.40,3.19,mpfi,136,...,99.4,front,4wd,sedan,4.0,std,gas,audi,164.0,2



Raw data saved to data/raw/raw_vehicles.csv


In [2]:
# Print the official target column specified by the creators
print("\nOfficial UCI Target:")
print(vehicle_data.metadata.target_col)


Official UCI Target:
['symboling']


In [ ]:
import numpy as np

# 1. Standardize Missing Values
# Replace all '?' with standard Python missing value tags (NaN)
df = df.replace('?', np.nan)

# Show which columns have missing data
print("--- Missing Data Before Cleaning ---")
print(df.isnull().sum()[df.isnull().sum() > 0])

# 2. Fix Data Types
# These columns contain numbers but if they were read as text because of the '?'
numeric_cols_to_fix = ['normalized-losses', 'bore', 'stroke', 'horsepower', 'peak-rpm', 'price']

for col in numeric_cols_to_fix:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# 3. Handle Missing Data (Imputation)
# For numeric columns: fill blanks with the median (middle value)
numeric_columns = df.select_dtypes(include=['float64', 'int64']).columns
df[numeric_columns]=df[numeric_columns].fillna(df[numeric_columns].median())

# For text/categorical columns: fill blanks with the mode (most common value)
categorical_columns = df.select_dtypes(include=['str']).columns
df[categorical_columns] = df[categorical_columns].fillna(df[categorical_columns].mode().iloc[0])

# 4. Remove Duplicates
duplicate_count = df.duplicated().sum()
print(f"\n--- Duplicates ---")
print(f"Found and dropped {duplicate_count} duplicate rows.")
df = df.drop_duplicates()

# 5. Verify the Cleanup
print("\n--- Final Cleaned Data Structure ---")
df.info()

In [8]:
# Outliers Detection
df.describe()

# Define the columns that require outlier treatment
numeric_columns = df.select_dtypes(include=['float64','int64']).columns

for col in numeric_columns:

    # Convert the column to float to safely hold decimal boundary values
    df[col] = df[col].astype('float64')

    # 1. Calculate the 25th (Q1) and 75th (Q3) percentiles
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    
    # 2. Calculate the Interquartile Range (IQR) and boundaries
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    print(f"end={col}: Lower Bound = {lower_bound}, Upper Bound = {upper_bound}")

    # 3. Cap the outliers using Pandas .loc
    # Overwrite values below the lower bound
    df.loc[df[col] < lower_bound, col] = lower_bound
    
    # Overwrite values above the upper bound
    df.loc[df[col] > upper_bound, col] = upper_bound

# 4. Verify the changes by printing the updated statistical summary
df.describe()


end=price: Lower Bound = -5280.0, Upper Bound = 29568.0
end=highway-mpg: Lower Bound = 11.5, Upper Bound = 47.5
end=city-mpg: Lower Bound = 2.5, Upper Bound = 46.5
end=peak-rpm: Lower Bound = 3750.0, Upper Bound = 6550.0
end=horsepower: Lower Bound = 1.0, Upper Bound = 185.0
end=compression-ratio: Lower Bound = 7.399999999999999, Upper Bound = 10.600000000000001
end=stroke: Lower Bound = 2.6599999999999993, Upper Bound = 3.8600000000000003
end=bore: Lower Bound = 2.505, Upper Bound = 4.2250000000000005
end=engine-size: Lower Bound = 31.0, Upper Bound = 207.0
end=num-of-cylinders: Lower Bound = 4.0, Upper Bound = 4.0
end=curb-weight: Lower Bound = 960.0, Upper Bound = 4120.0
end=height: Lower Bound = 46.75, Upper Bound = 60.75
end=width: Lower Bound = 59.89999999999998, Upper Bound = 71.10000000000002
end=length: Lower Bound = 141.10000000000002, Upper Bound = 208.29999999999995
end=wheel-base: Lower Bound = 82.64999999999999, Upper Bound = 114.25000000000001
end=num-of-doors: Lower Bou

,price,highway-mpg,city-mpg,peak-rpm,horsepower,compression-ratio,stroke,bore,engine-size,num-of-cylinders,curb-weight,height,width,length,wheel-base,num-of-doors,normalized-losses,symboling
count,205.000000,205.000000,205.000000,205.000000,205.000000,205.000000,205.000000,205.000000,205.000000,205.0,205.000000,205.000000,205.000000,205.000000,205.000000,205.000000,205.000000,205.000000
mean,12713.297561,30.680488,25.204878,5125.609756,102.892683,9.039122,3.262488,3.329366,124.570732,4.0,2555.565854,53.724878,65.884390,174.049268,98.710976,3.131707,119.990244,0.834146
std,6599.964024,6.682147,6.491659,475.544315,35.564608,0.793356,0.276881,0.270858,33.974343,0.0,520.680204,2.443522,2.083114,12.337289,5.882586,0.993715,29.931784,1.245307
min,5118.000000,16.000000,13.000000,4150.000000,48.000000,7.400000,2.660000,2.540000,61.000000,4.0,1488.000000,47.800000,60.300000,141.100000,86.600000,2.000000,65.000000,-2.000000
25%,7788.000000,25.000000,19.000000,4800.000000,70.000000,8.600000,3.110000,3.150000,97.000000,4.0,2145.000000,52.000000,64.100000,166.300000,94.500000,2.000000,101.000000,0.000000
50%,10295.000000,30.000000,24.000000,5200.000000,95.000000,9.000000,3.290000,3.310000,120.000000,4.0,2414.000000,54.100000,65.500000,173.200000,97.000000,4.000000,115.000000,1.000000
75%,16500.000000,34.000000,30.000000,5500.000000,116.000000,9.400000,3.410000,3.580000,141.000000,4.0,2935.000000,55.500000,66.900000,183.100000,102.400000,4.000000,137.000000,2.000000
max,29568.000000,47.500000,46.500000,6550.000000,185.000000,10.600000,3.860000,3.940000,207.000000,4.0,4066.000000,59.800000,71.100000,208.100000,114.250000,4.000000,191.000000,3.000000


In [9]:
import os

# Create the processed directory if it doesn't exist
os.makedirs('../data/processed', exist_ok=True)

# Save the cleaned DataFrame without the index column
df.to_csv('../data/processed/cleaned_vehicles.csv', index=False)
print("Cleaned dataset successfully saved to data/processed/cleaned_vehicles.csv")

Cleaned dataset successfully saved to data/processed/cleaned_vehicles.csv
